In [10]:
import random
import nest_asyncio
nest_asyncio.apply()

import os 
from dotenv import load_dotenv
from pydantic_ai import Agent, RunContext
from dataclasses import dataclass

load_dotenv()

True

````markdown
# Registering Tools in PydanticAI

PydanticAI provides multiple ways to register tools with an agent depending on whether the tool needs access to the agent context.

---

# 1. Using `@agent.tool`

Use `@agent.tool` when your tool needs access to the agent context (`RunContext`).

## Example

```python
from pydantic_ai import Agent, RunContext

agent = Agent("openai:gpt-4o")

@agent.tool
def get_user_name(ctx: RunContext[str]) -> str:
    return f"Current user: {ctx.deps}"
````

## When to Use

Use this approach when:

* The tool needs dependency injection
* The tool requires runtime context
* The tool needs access to shared state or configuration

---

# 2. Using `@agent.tool_plain`

Use `@agent.tool_plain` for simple tools that do not require access to the agent context.

## Example

```python
from pydantic_ai import Agent

agent = Agent("openai:gpt-4o")

@agent.tool_plain
def add_numbers(a: int, b: int) -> int:
    return a + b
```

## When to Use

Use this approach when:

* The tool is self-contained
* No context or dependencies are required
* The function behaves like a regular utility function

---

# 3. Using the `tools` Argument in `Agent`

You can also register tools directly when creating the agent using the `tools` parameter.

The `tools` argument accepts:

* Plain functions
* `Tool` instances

## Example with Plain Functions

```python
from pydantic_ai import Agent

def multiply(a: int, b: int) -> int:
    return a * b

agent = Agent(
    "openai:gpt-4o",
    tools=[multiply]
)
```

## Example with `Tool` Instances

```python
from pydantic_ai import Agent, Tool

def divide(a: float, b: float) -> float:
    return a / b

divide_tool = Tool(divide)

agent = Agent(
    "openai:gpt-4o",
    tools=[divide_tool]
)
```

---

# Summary

| Method              | Context Access | Best For                      |
| ------------------- | -------------- | ----------------------------- |
| `@agent.tool`       | Yes            | Context-aware tools           |
| `@agent.tool_plain` | No             | Simple utility tools          |
| `tools=[...]`       | Optional       | Centralized tool registration |

---

# Recommendation

* Use `@agent.tool` for advanced workflows requiring context or dependencies.
* Use `@agent.tool_plain` for lightweight helper functions.
* Use the `tools` argument when organizing tools separately from the agent definition.

```
```


In [111]:
from dataclasses import dataclass
from typing import Dict

@dataclass
class product_info:
    product: Dict[str,Dict[str,int]]
    
products = product_info(
    product={
        "Wireless Mouse": {
            "price": 1200,
            "discount": 10,
            "stock_available": 50
        },

        "Mechanical Keyboard": {
            "price": 3500,
            "discount": 15,
            "stock_available": 25
        },

        "USB-C Charger": {
            "price": 1800,
            "discount": 5,
            "stock_available": 100
        },

        "Gaming Headset": {
            "price": 4500,
            "discount": 20,
            "stock_available": 15
        },

        "laptop": {
            "price": 72200,
            "discount": 12,
            "stock_available": 40
        }
    }
)

print(products)

product_info(product={'Wireless Mouse': {'price': 1200, 'discount': 10, 'stock_available': 50}, 'Mechanical Keyboard': {'price': 3500, 'discount': 15, 'stock_available': 25}, 'USB-C Charger': {'price': 1800, 'discount': 5, 'stock_available': 100}, 'Gaming Headset': {'price': 4500, 'discount': 20, 'stock_available': 15}, 'laptop': {'price': 72200, 'discount': 12, 'stock_available': 40}})


In [105]:
amazon_agent = Agent(
                     "groq:openai/gpt-oss-120b",
                     description="This is an  chatbot which can help users find the products that they need",
                     system_prompt="You are an amazon agent, your job is to help the customer " \
                     "find the product they need, " \
                     "also you have to answer any questions they might have about the products"
                     "When the user asks for a coupon, discount code, promo code, or offer code, " \
                     "you MUST call the create_couponcode tool. Do not invent coupon codes." \
                     "Do not invent extra products, information"\
                     "First check whether the information exists, if not then use the create_couponcode tool"\
                     "Dont tell the user about the create_couponcode function that you will be using in the background to generate the coupon code")

@amazon_agent.tool_plain
def create_couponcode() -> str:
    """Generate a random 10-letter coupon code."""
    return "".join(random.choices("ABCDEFGHIJKLMNOPQRSTUVWXYZ", k=10))
    

@amazon_agent.tool
def help_with_context(ctx: RunContext[product_info], item_name: str) -> str:
    """Check product price, discount, and stock availability by product name."""
    
    item = ctx.deps.product.get(item_name)

    if item is None:
        return f"{item_name} is not available in the product database."

    return (
        f"{item_name} is available. "
        f"Price: {item['price']}. "
        f"Discount: {item['discount']}%. "
        f"Stock available: {item['stock_available']}."
    )



# print(create_couponcode())

In [113]:
agent_response = amazon_agent.run_sync(
    "I want to shop for somethings, do you have any discount?",
    deps=products
)

print(agent_response.output)

Sure! I’ve generated a discount code you can use at checkout:

**BAFDZTSLRX**

Just enter this code during purchase to apply the discount.

What are you looking to shop for today? Let me know the product name or category, and I’ll help you find the perfect item!


In [118]:
agent_response = amazon_agent.run_sync(
    "I want to shop for a laptop, do you have any discount?",
    deps=products
)

print(agent_response.output)

Yes! The laptop you’re interested in is currently priced at ₹72,200 and comes with a **12% discount**. We have 40 units in stock, so it’s ready for you to purchase. Let me know if you’d like any more details or help with placing an order!
